# Project 1: Automatic Review Analyzer

This notebook applies the Unit 1 linear-classification ideas to the complete 3,000-sentence UCI Sentiment Labelled Sentences dataset.

Workflow: **reviews → reproducible split → sparse `X` + `y` → controlled comparison → lambda selection → final test comparison → word weights → separate two-feature visualizations.**

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'project_1':
    candidate = PROJECT_DIR / 'unit_1' / 'project_1'
    if candidate.exists(): PROJECT_DIR = candidate
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
from linear_classification import accuracy, average_perceptron, pegasos, perceptron
from review_data import prepare_review_data, tokenize

## 1. Dataset and reproducible split

The dataset contains 1,000 IMDb, 1,000 Amazon, and 1,000 Yelp sentences: 3,000 reviews total, balanced between positive and negative labels. A fixed seed gives a stratified 70/15/15 split. The test set remains untouched until final evaluation.

In [ ]:
SEED = 42
data = prepare_review_data(PROJECT_DIR, seed=SEED, min_count=2)
reviews = data['reviews']; train_reviews = data['train_reviews']; validation_reviews = data['validation_reviews']; test_reviews = data['test_reviews']
vocabulary = data['vocabulary']
X_train, y_train = data['X_train'], data['y_train']
X_validation, y_validation = data['X_validation'], data['y_validation']
X_test, y_test = data['X_test'], data['y_test']
train_data = list(zip(y_train, X_train)); validation_data = list(zip(y_validation, X_validation)); test_data = list(zip(y_test, X_test))
print(f'Total: {len(reviews)} | Positive: {sum(r[0] == 1 for r in reviews)} | Negative: {sum(r[0] == -1 for r in reviews)}')
print(f'Train: {len(train_data)} | Validation: {len(validation_data)} | Test: {len(test_data)}')
print(f'Vocabulary: {len(vocabulary)}')

## 2. Review text → sparse `X` and `y`

Review-specific tokenization, vocabulary construction, and vectorization live in `review_data.py`. The vocabulary is learned from training reviews only, then reused for validation and test reviews. The representation is a sparse binary bag of words.

In [ ]:
print('Example tokens:', tokenize(train_reviews[0][1])[:12])
print('Example sparse vector:', list(X_train[0].items())[:10])
print('X_train:', len(X_train), '| y_train:', len(y_train))

## 3. Controlled comparison

This asks how the three learning rules compare with the same training data and epoch budget. Pegasos uses fixed `lambda_=1e-3` here. This experiment is intentionally separate from later tuning.

In [ ]:
EPOCHS = 10; BATCH_SIZE = 32; FIXED_LAMBDA = 1e-3
def timed(train_fn):
    start = time.perf_counter(); weights = train_fn(); return weights, time.perf_counter() - start
specs = {
    'Perceptron': lambda: perceptron(train_data, epochs=EPOCHS),
    'Average Perceptron': lambda: average_perceptron(train_data, epochs=EPOCHS),
    'Pegasos (lambda=1e-3)': lambda: pegasos(train_data, lambda_=FIXED_LAMBDA, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED),
}
controlled_models = {}
for name, train_fn in specs.items():
    weights, elapsed = timed(train_fn); controlled_models[name] = weights
    print(f'{name:25} | {elapsed:.4f}s | validation={accuracy(weights, validation_data):.2%}')

## 4. Pegasos hyperparameter selection

Now select the regularization value using validation accuracy only.

$$
\lambda^*=\arg\max_{\lambda}\mathrm{ValidationAccuracy}(\lambda).
$$

If several candidates tie, the smallest value is selected.

In [ ]:
lambda_candidates = [1e-6, 1e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1]
lambda_results = []
for lambda_ in lambda_candidates:
    weights, elapsed = timed(lambda: pegasos(train_data, lambda_=lambda_, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED))
    score = accuracy(weights, validation_data); lambda_results.append((lambda_, score, elapsed, weights))
    print(f'lambda={lambda_:8.1e} | validation={score:.2%} | time={elapsed:.4f}s')
best_score = max(row[1] for row in lambda_results)
best_lambda = min(row[0] for row in lambda_results if row[1] == best_score)
print(f'\nSelected lambda*: {best_lambda:.1e} | validation accuracy={best_score:.2%}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogx([r[0] for r in lambda_results], [r[1] for r in lambda_results], marker='o')
plt.axvline(best_lambda, linestyle='--', label=f'lambda* = {best_lambda:.1e}')
plt.xlabel('Pegasos lambda'); plt.ylabel('Validation accuracy'); plt.title('Pegasos regularization selection'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Final model comparison

The controlled comparison uses fixed lambda. The tuned comparison uses the selected `lambda*`. Perceptron and Average Perceptron have no corresponding regularization hyperparameter. After selection, the final models are retrained on train + validation and evaluated once on the untouched test set.

In [ ]:
final_train_data = train_data + validation_data
final_models = {
    'Perceptron': perceptron(final_train_data, epochs=EPOCHS),
    'Average Perceptron': average_perceptron(final_train_data, epochs=EPOCHS),
    'Pegasos': pegasos(final_train_data, lambda_=best_lambda, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED),
}
for name, weights in final_models.items():
    print(f'{name:20} | train+validation={accuracy(weights, final_train_data):.2%} | test={accuracy(weights, test_data):.2%}')

## 6. Strongest learned words

A positive weight pushes the score toward the positive class and a negative weight pushes it toward the negative class. The actual feature contribution is `theta_j * x_j`; with binary bag-of-words, a present word has `x_j = 1`, so its weight is its contribution to the score.

In [ ]:
index_to_word = {index: word for word, index in vocabulary.items()}
for name, weights in final_models.items():
    positive = sorted(((weight, index_to_word[index]) for index, weight in weights.items() if weight > 0), reverse=True)[:15]
    negative = sorted(((weight, index_to_word[index]) for index, weight in weights.items() if weight < 0))[:15]
    print(f'\n{name}')
    print('Strongest positive words:', [(word, round(weight, 4)) for weight, word in positive])
    print('Strongest negative words:', [(word, round(weight, 4)) for weight, word in negative])

## 7. Separate two-feature decision-boundary visualizations

The complete bag-of-words classifier has thousands of dimensions. To show the geometry clearly, we train separate two-feature models using `excellent` and `terrible` as coordinates. Each classifier gets its own figure, so the plots do not imply that the full sentiment model is two-dimensional.

In [ ]:
FEATURE_WORDS = ('excellent', 'terrible')
feature_indices = [vocabulary.get(word) for word in FEATURE_WORDS]
if any(index is None for index in feature_indices): raise ValueError('selected visualization words must be in the training vocabulary')
def project_two_features(rows):
    return [(label, {j: float(features.get(index, 0.0)) for j, index in enumerate(feature_indices)}) for label, features in rows]
two_d_train = project_two_features(final_train_data)
two_d_models = {
    'Perceptron': perceptron(two_d_train, epochs=EPOCHS),
    'Average Perceptron': average_perceptron(two_d_train, epochs=EPOCHS),
    'Pegasos': pegasos(two_d_train, lambda_=best_lambda, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED),
}
x_values = np.array([features[0] for _, features in two_d_train]); y_values = np.array([features[1] for _, features in two_d_train]); labels = np.array([label for label, _ in two_d_train]); x_line = np.linspace(-0.1, 1.1, 100)
for name, weights in two_d_models.items():
    plt.figure(figsize=(7, 5))
    plt.scatter(x_values[labels == 1], y_values[labels == 1], marker='o', label='positive')
    plt.scatter(x_values[labels == -1], y_values[labels == -1], marker='x', label='negative')
    if abs(weights.get(1, 0.0)) > 1e-12:
        plt.plot(x_line, -(weights.get(0, 0.0) / weights.get(1, 0.0)) * x_line, label='decision boundary')
    plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1); plt.xlabel('excellent'); plt.ylabel('terrible'); plt.title(f'{name}: two-feature decision boundary'); plt.legend(); plt.tight_layout(); plt.show()

## 8. Interpretation

Perceptron and Average Perceptron also learn a weight for every vocabulary feature; learned word weights are not unique to Pegasos. Pegasos additionally uses L2 regularization, which controls parameter magnitude.

A fixed lambda is appropriate for a controlled comparison, while the selected `lambda*` is appropriate for the tuned final Pegasos model. Validation data drives selection; the test set is reserved for final evaluation.

Dataset source: Kotzias et al., **Sentiment Labelled Sentences**, UCI Machine Learning Repository, DOI 10.24432/C57604.